In [1]:
from sklearn.linear_model import LinearRegression
import numpy as np
import os
import pandas as pd
import requests, json


In [2]:
def train_test_split(X, y, split = 0.3):
    length = int(min(len(X), len(y)) * (1 - split))
    X_train = X[:length]
    X_test = X[length:]
    y_train = y[:length]
    y_test = y[length:]
    return X_train, X_test, y_train, y_test


In [3]:
def process_data(df, n = 1):
    df = df.copy()
    df = df.sort_values(by="Timestamp")
    # drop duplicate timestamps
    df = df.drop_duplicates(subset=["Timestamp"])

    # generate lags
    max_lag = 10
    for i in range(1, max_lag):
        df[f"lag_{i}"] = df["Low Price"].shift(i)



    # add next value that needs to be predicted
    df["NextValue"] = df["Low Price"].shift(-n)

    df = df.dropna()

    # create X
    drop = ["Closing Price", "Opening Price", "High Price", "Timestamp", "No of Shares", "NextValue"]
    X = df.drop(columns=drop, errors='ignore')

    y = df["NextValue"]

    return X, y

In [4]:

def get_data(ticker, period = "h6"):
    url = f"https://tornsy.com/api/{ticker}?interval={period}"
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch data for {ticker}. Status code: {response.status_code}")
    data = response.json()
    # print(data)
    df = pd.DataFrame(data["data"], columns=["Timestamp", "Opening Price", "Closing Price", "High Price", "Low Price", "No of Shares"])
    return df


In [6]:
stockNames = [
    "TSB",
    "TCI",
    "SYS",
    "LAG",
    "IOU",
    "GRN",
    "THS",
    "YAZ",
    "TCT",
    "CNC",
    "MSG",
    "TMI",
    "TCP",
    "IIL",
    "FHG",
    "SYM",
    "LSC",
    "PRN",
    "EWM",
    "TCM",
    "ELT",
    "HRG",
    "TGP",
    "MUN",
    "WSU",
    "IST",
    "BAG",
    "EVL",
    "MCS",
    "WLT",
    "TCC",
    "ASS",
    "CBD",
    "LOS",
    "PTS"
]

models = {}
period = "h6"

for stock in stockNames:
    df = get_data(stock, period)
    # print(df.head())
    X, y = process_data(df, n = 4)
    # print(X.head())
    X_train, X_test, y_train, y_test = train_test_split(X, y)
    temp_model = LinearRegression()
    temp_model.fit(X_train, y_train)
    score = temp_model.score(X_test, y_test)
    print(f"Model for {stock} has a score of {score:.4f}")
    models[stock] = (temp_model, score)

Model for TSB has a score of 0.7197
Model for TCI has a score of 0.8114
Model for SYS has a score of 0.6779
Model for LAG has a score of 0.9120
Model for IOU has a score of 0.6498
Model for GRN has a score of 0.9052
Model for THS has a score of 0.5878
Model for YAZ has a score of 0.8689
Model for TCT has a score of 0.4479
Model for CNC has a score of 0.7587
Model for MSG has a score of 0.8691
Model for TMI has a score of 0.7785
Model for TCP has a score of 0.7180
Model for IIL has a score of 0.9534
Model for FHG has a score of 0.8911
Model for SYM has a score of 0.8220
Model for LSC has a score of 0.8286
Model for PRN has a score of 0.8666
Model for EWM has a score of 0.8248
Model for TCM has a score of 0.8985
Model for ELT has a score of 0.8394
Model for HRG has a score of 0.8248
Model for TGP has a score of 0.9131
Model for MUN has a score of 0.8185
Model for WSU has a score of 0.6143
Model for IST has a score of 0.8678
Model for BAG has a score of 0.8975
Model for EVL has a score of

In [ ]:
files_path = "./stocksData"
files = os.listdir(files_path)
print(files)
data_files = {}
stocks = []

for file in files:
    if file.endswith(".csv"):
        with open(files_path + "/" + file, "r") as f:
            df = pd.read_csv(f)
            if "Low Price" in df.columns:
                print(f"Processing file: {file}")
                data_name = file.replace(".csv", "")
                stocks.append(data_name)
                try:
                    X, y = process_data(df)
                    data_files[data_name] = (X, y)
                except Exception as e:
                    print(f"Error processing {file}: {e}")




['ass_h6.csv', 'bag_h6.csv', 'cbd_h6.csv', 'cnc_h6.csv', 'elt_h6.csv', 'evl_h6.csv', 'ewm_h6.csv', 'fhg_h6.csv', 'grn_h6.csv', 'hrg_h6.csv', 'iil_h6.csv', 'iou_h6.csv', 'ist_h6.csv', 'lag_h6.csv', 'los_h6.csv', 'lsc_h6.csv', 'mcs_h6.csv', 'msg_h6.csv', 'mun_h6.csv', 'prn_h6.csv', 'pts_h6.csv', 'sym_h6.csv', 'sys_h6.csv', 'tcc_h6.csv', 'tci_h6.csv', 'tcm_h6.csv', 'tcp_h6.csv', 'tct_h6.csv', 'tgp_h6.csv', 'ths_h6.csv', 'tmi_h6.csv', 'tsb_h6.csv', 'wlt_h6.csv', 'wsu_h6.csv', 'yaz_h6.csv']
Processing file: ass_h6.csv
Processing file: bag_h6.csv
Processing file: cbd_h6.csv
Processing file: cnc_h6.csv
Processing file: elt_h6.csv
Processing file: evl_h6.csv
Processing file: ewm_h6.csv
Processing file: fhg_h6.csv
Processing file: grn_h6.csv
Processing file: hrg_h6.csv
Processing file: iil_h6.csv
Processing file: iou_h6.csv
Processing file: ist_h6.csv
Processing file: lag_h6.csv
Processing file: los_h6.csv
Processing file: lsc_h6.csv
Processing file: mcs_h6.csv
Processing file: msg_h6.csv
Proce

In [ ]:
for d in data_files:
    print(d, data_files[d][1].head())

ass_h6 16993     86.40
58938    305.47
11003    491.38
28980    189.30
76917    216.64
Name: NextValue, dtype: float64
bag_h6 5009     483.91
28978    279.29
70923    144.58
98569     88.55
58941    143.92
Name: NextValue, dtype: float64
cbd_h6 82905     278.00
70922     209.56
146205    279.46
70924     396.48
157864     90.33
Name: NextValue, dtype: float64
cnc_h6 11001    292.93
5010     744.33
22987    747.54
22988    364.59
16997    493.38
Name: NextValue, dtype: float64
elt_h6 5009     759.16
22986    429.16
46955    747.54
22988    429.54
46957    581.53
Name: NextValue, dtype: float64
evl_h6 11001    370.55
16994    290.11
5011     301.73
11004    292.16
5013     292.84
Name: NextValue, dtype: float64
ewm_h6 82905     483.91
28978     305.90
177200    279.46
70924     429.54
46957     146.26
Name: NextValue, dtype: float64
fhg_h6 34969    483.91
28978    587.26
34971    747.54
22988    587.57
34973    751.71
Name: NextValue, dtype: float64
grn_h6 76913    483.91
28978    744.33

In [ ]:
models = {}

for data_name in data_files:
    X, y = data_files[data_name]
    X_train, X_test, y_train, y_test = train_test_split(X, y)

    model = LinearRegression()
    model.fit(X_train, y_train)

    score = model.score(X_test, y_test)
    print(f"Model for {data_name} has a score of: {score}")

    models[data_name] = model

Model for ass_h6 has a score of: -0.04432484115147184
Model for bag_h6 has a score of: -0.04247456718151632
Model for cbd_h6 has a score of: -0.025081171066833052
Model for cnc_h6 has a score of: -0.15888706219701798
Model for elt_h6 has a score of: -0.08553885338081768
Model for evl_h6 has a score of: -0.12559090801407247
Model for ewm_h6 has a score of: -0.026259471351535835
Model for fhg_h6 has a score of: -0.3391904926749212
Model for grn_h6 has a score of: -0.0890389307257824
Model for hrg_h6 has a score of: -0.049151341300584184
Model for iil_h6 has a score of: -0.029429049877150337
Model for iou_h6 has a score of: -0.020621174734328562
Model for ist_h6 has a score of: -0.0338688557773954
Model for lag_h6 has a score of: 0.9950109549706806
Model for los_h6 has a score of: -0.02537177927620138
Model for lsc_h6 has a score of: -0.032190754143132105
Model for mcs_h6 has a score of: -0.016169585895896788
Model for msg_h6 has a score of: -0.026707734186897758
Model for mun_h6 has a sc

In [ ]:
# check lstm
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


def create_lstm_model(input_shape):
    lstm_model = Sequential()
    lstm_model.add(LSTM(50, return_sequences=True, input_shape= input_shape))
    lstm_model.add(Dropout(0.2))
    lstm_model.add(Dense(1))
    lstm_model.compile(optimizer='adam', loss='mean_squared_error')
    return lstm_model

# add early stoppage

In [ ]:
lstm_models = {}
for data_name in data_files:
    X, y = data_files[data_name]
    X_train, X_test, y_train, y_test = train_test_split(X.values.reshape(-1, X.shape[1], 1), y.values)

    lstm_model = create_lstm_model((X_train.shape[1], 1))
    early_stopping = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)
    lstm_model.fit(X_train, y_train, epochs=500, batch_size=32, verbose=1, callbacks = [early_stopping])
    score = lstm_model.evaluate(X_test, y_test, verbose=0)
    print(f"LSTM Model for {data_name} has a score of: {score}")

    lstm_models[data_name] = lstm_model

Epoch 1/500


131/131 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 235532.0000
Epoch 2/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 225368.7031
Epoch 3/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 220980.8438
Epoch 4/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 218249.7188
Epoch 5/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 209374.7031
Epoch 6/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 198315.4844
Epoch 7/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 192676.8594
Epoch 8/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 192842.6406
Epoch 9/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 194164.4688
Epoch 10/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 181301.0469
Epoch 11/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 184151.6719
Epoch 12/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 176802.4844
Epoch 13/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 169992.2344
Epoch 14/500
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - los

KeyboardInterrupt: 

In [ ]:
# score each lstm model

for data in data_files:
    X, y = data_files[data]
    X_test, y_test = X.values.reshape(-1, X.shape[1], 1), y.values
    lstm_model = lstm_models[data]
    score = lstm_model.evaluate(X_test, y_test, verbose=0)
    print(f"LSTM Model for {data} has a score of: {score}")

LSTM Model for ass_h6 has a score of: 66753.6171875
LSTM Model for bag_h6 has a score of: 56173.4609375
LSTM Model for cbd_h6 has a score of: 58246.08203125
LSTM Model for cnc_h6 has a score of: 33302.53125
LSTM Model for elt_h6 has a score of: 55495.28515625
LSTM Model for evl_h6 has a score of: 9975.3349609375
LSTM Model for ewm_h6 has a score of: 63571.6640625


KeyError: 'fhg_h6'

# Saving the model: models is a dict with modelName: (model, score)

In [ ]:
# save models
import pickle

for modelName in models:
    model, score = models[modelName]
    if score < 0.90:
        continue
    filename = modelName + "_" + period + ".pkl"
    location = "./models/"
    final_location = os.path.join(location, filename)

    with open(final_location, 'wb') as file:
        pickle.dump(model, file)
    print(f"Model for {modelName} saved to {final_location}")


Model for TSB saved to ./models/TSB_h1.pkl
Model for TCI saved to ./models/TCI_h1.pkl
Model for SYS saved to ./models/SYS_h1.pkl
Model for LAG saved to ./models/LAG_h1.pkl
Model for IOU saved to ./models/IOU_h1.pkl
Model for GRN saved to ./models/GRN_h1.pkl
Model for THS saved to ./models/THS_h1.pkl
Model for YAZ saved to ./models/YAZ_h1.pkl
Model for TCT saved to ./models/TCT_h1.pkl
Model for CNC saved to ./models/CNC_h1.pkl
Model for MSG saved to ./models/MSG_h1.pkl
Model for TMI saved to ./models/TMI_h1.pkl
Model for TCP saved to ./models/TCP_h1.pkl
Model for IIL saved to ./models/IIL_h1.pkl
Model for FHG saved to ./models/FHG_h1.pkl
Model for SYM saved to ./models/SYM_h1.pkl
Model for LSC saved to ./models/LSC_h1.pkl
Model for PRN saved to ./models/PRN_h1.pkl
Model for EWM saved to ./models/EWM_h1.pkl
Model for TCM saved to ./models/TCM_h1.pkl
Model for ELT saved to ./models/ELT_h1.pkl
Model for HRG saved to ./models/HRG_h1.pkl
Model for TGP saved to ./models/TGP_h1.pkl
Model for M

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2991, 11), (2991, 11), (2991,), (2991,))

In [ ]:
X_train, y_train

(      Low Price   lag_1   lag_2   lag_3   lag_4   lag_5   lag_6   lag_7  \
 9        448.14  448.46  448.00  448.25  448.17  447.72  446.90  445.61   
 10       448.60  448.14  448.46  448.00  448.25  448.17  447.72  446.90   
 11       448.63  448.60  448.14  448.46  448.00  448.25  448.17  447.72   
 12       449.65  448.63  448.60  448.14  448.46  448.00  448.25  448.17   
 13       450.16  449.65  448.63  448.60  448.14  448.46  448.00  448.25   
 ...         ...     ...     ...     ...     ...     ...     ...     ...   
 2995     441.99  441.76  441.39  440.85  440.98  440.77  440.56  440.07   
 2996     442.55  441.99  441.76  441.39  440.85  440.98  440.77  440.56   
 2997     442.24  442.55  441.99  441.76  441.39  440.85  440.98  440.77   
 2998     442.00  442.24  442.55  441.99  441.76  441.39  440.85  440.98   
 2999     441.72  442.00  442.24  442.55  441.99  441.76  441.39  440.85   
 
        lag_8   lag_9  target  
 9     444.79  444.70  448.60  
 10    445.61  444.79 

In [ ]:
X_train.columns

Index(['Low Price', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6',
       'lag_7', 'lag_8', 'lag_9'],
      dtype='object')

In [ ]:
import requests, json, math
import pandas as pd
import numpy as np

def getStockData(stockTicker, to=""):
    url = f"https://tornsy.com/api/{stockTicker}?interval=h6&to={to}"
    print(f"Making request to {url}")
    response = requests.get(url)
    print(f"Response: {response.status_code}")

    if response.status_code != 200:
        print(f"Error fetching data: {response.text}")
        return None

    try:
        jsonResponse = json.loads(response.text)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return None

    if "data" in jsonResponse and len(jsonResponse["data"]) > 0:
        df = pd.DataFrame(jsonResponse["data"], columns=["Timestamp", "Opening Price", "High Price", "Low Price", "Closing Price", "No of Shares"])
        numerical_cols = ["Opening Price", "High Price", "Low Price", "Closing Price", "No of Shares"]
        for col in numerical_cols:
            df[col] = pd.to_numeric(df[col])
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], unit='s')
        df = df.sort_values(by="Timestamp")
        max_lag = 10
        for i in range(1, max_lag):
            df[f"lag_{i}"] = df["Low Price"].shift(i)
        df = df.tail(1)
        drop_cols = ["Closing Price", "Opening Price", "High Price", "Timestamp", "No of Shares"]
        X = df.drop(columns=drop_cols, errors='ignore')
        X = X.dropna()

        if X.empty:
            print("Not enough data to create features for prediction.")
            return None

        return X
    else:
        print("No data found for the specified ticker.")
        return None

def predict(stockTicker):
    X_pred = getStockData(stockTicker)
    if X_pred is not None:
        model_key = "lag" + "_h6"
        if model_key in models:
            model = models[model_key]
            prediction = model.predict(X_pred)
            current = X_pred["Low Price"].values[0]
            # print(f"Prediction for {stockTicker}: {prediction[0]}")

            change = (prediction[0] - current) / current * 100
            # print(f"Change: {change:.2f}%")
            return prediction[0], change
        else:
            print(f"No trained model found for the key '{model_key}'.")
            return None, None
    else:
        print("Could not get data for prediction.")
        return None

In [ ]:

for stock in stocks:
  pred, change = predict(stock)
  print(f"Prediction: {pred}, Change: {change}")

Making request to https://tornsy.com/api/lag?interval=h6&to=
Response: 200
Prediction for lag: 450.36032176883117
Change: -0.01%
Prediction: 450.36032176883117, Change: -0.011029557773773858
